# Module 3 - Session 5: RAG Evaluation
Goal: Measure our RAG pipeline using the RAG Triad —
Context Relevance, Groundedness, and Answer Relevance.
We use LLM-as-Judge to score each dimension from 0 to 1.

In [ ]:
# Install all libraries needed for this session
# No new libraries today — everything is from previous sessions
# sentence-transformers — convert text to vectors
# faiss-cpu — vector database
# pymupdf — read PDF
# groq — LLM for RAG answers AND for LLM-as-Judge scoring
!pip install -q sentence-transformers faiss-cpu pymupdf groq

## Step 2 - Rebuild RAG Pipeline
We rebuild the full pipeline from Session 4 in one cell.
Sentence chunking strategy — our proven winner.
This notebook is self contained and runs on its own.

In [ ]:
import os
import json
import numpy as np
import faiss
import fitz
from groq import Groq
from google.colab import userdata
from sentence_transformers import SentenceTransformer

# Load API key and clients
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq()

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Recreate Swiggy policy PDF
policy_sections = [
    {
        "title": "Section 1: Delivery Partner Eligibility",
        "content": "To become a Swiggy delivery partner, you must be at least 18 years old. You must own a smartphone with Android 6.0 or above. A valid driving license and vehicle registration certificate are mandatory. Two-wheeler or bicycle ownership is required for deliveries. You must have a valid Aadhaar card and PAN card for verification. Background verification will be conducted before onboarding. Partners must complete the Swiggy training module before first delivery."
    },
    {
        "title": "Section 2: Earnings and Payments",
        "content": "Delivery partners earn a base fee for every order delivered. Additional incentives are provided for completing surge orders during peak hours. Weekly payments are credited directly to the registered bank account. Partners can track their daily and weekly earnings in the Swiggy Partner app. Surge bonuses are calculated based on orders completed in high demand zones. Tips provided by customers are credited in full to the delivery partner. Payment disputes must be raised within 7 days through the partner support portal."
    },
    {
        "title": "Section 3: Order Handling and Delivery Rules",
        "content": "Delivery partners must pick up orders within 5 minutes of reaching the restaurant. Orders must be delivered within the estimated time shown in the app. Partners must not open or tamper with food packaging at any time. If a customer is unreachable, partners must wait 5 minutes before contacting support. Proof of delivery photo must be uploaded for all orders above Rs 500. Partners must follow the route suggested by the Swiggy navigation system. Any delivery delay beyond 15 minutes must be reported through the app."
    },
    {
        "title": "Section 4: Code of Conduct",
        "content": "Delivery partners must maintain professional behavior with customers at all times. Use of abusive language or aggressive behavior will result in immediate suspension. Partners must wear the Swiggy delivery uniform and helmet during all deliveries. Mobile phones must not be used while riding the delivery vehicle. Partners must not accept orders outside the Swiggy platform. Any form of fraud including fake deliveries will result in permanent deactivation. Partners must report any accident or incident immediately through the app."
    },
    {
        "title": "Section 5: Grievance and Support",
        "content": "Delivery partners can raise grievances through the Swiggy Partner app support section. All grievances will be acknowledged within 24 hours of submission. Resolution of grievances will be provided within 5 working days. Partners can escalate unresolved issues to the regional partner support team. Swiggy provides accident insurance coverage for all active delivery partners. Medical emergencies during delivery must be reported within 48 hours. Partners have the right to appeal any suspension or deactivation decision."
    }
]

# Create PDF
doc = fitz.open()
for section in policy_sections:
    page = doc.new_page()
    page.insert_text((50, 50), section["title"], fontsize=14, color=(0, 0, 0))
    rect = fitz.Rect(50, 80, 550, 750)
    page.insert_textbox(rect, section["content"], fontsize=10, color=(0, 0, 0))

pdf_path = "/content/swiggy_delivery_policy.pdf"
doc.save(pdf_path)
doc.close()

# Extract full text
doc = fitz.open(pdf_path)
full_text = ""
for page_num in range(len(doc)):
    full_text += doc[page_num].get_text()
doc.close()

# Sentence chunking — our winning strategy from Session 4
sentences = full_text.split(". ")
chunks = [s.strip().replace("\n", " ") for s in sentences if len(s.strip()) > 20]

# Build FAISS index
embeddings = model.encode(chunks)
embeddings_float = np.array(embeddings).astype("float32")
faiss.normalize_L2(embeddings_float)
index = faiss.IndexFlatIP(384)
index.add(embeddings_float)

print(f"Pipeline ready!")
print(f"Total chunks: {len(chunks)}")
print(f"FAISS index size: {index.ntotal}")

## Step 3 - LLM-as-Judge Scoring Functions
We build one function per RAG Triad dimension.
Each function sends a pair of texts to the LLM and asks for a score from 0 to 1.
The LLM returns a JSON with the score and a reason.

In [ ]:
def score_context_relevance(question, retrieved_chunk):
    """
    Score 1 — Context Relevance
    Did retrieval find the RIGHT chunk for this question?
    Input: question + retrieved chunk
    Output: score 0 to 1
    """
    prompt = f"""You are an expert RAG pipeline evaluator.

Score how relevant the retrieved context is to the question.

Question: {question}
Retrieved Context: {retrieved_chunk}

Return ONLY a JSON object in this exact format:
{{"score": 0.0, "reason": "one line explanation"}}

Score guide:
1.0 = context directly answers the question
0.7 = context is related but not a direct answer
0.4 = context is loosely related
0.0 = context is completely unrelated"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    # Parse the JSON response
    result = json.loads(response.choices[0].message.content)
    return result


def score_groundedness(retrieved_chunk, llm_answer):
    """
    Score 2 — Groundedness
    Did the LLM answer using ONLY the retrieved chunk?
    Input: retrieved chunk + LLM answer
    Output: score 0 to 1
    """
    prompt = f"""You are an expert RAG pipeline evaluator.

Score how well the answer is supported by the context.
A grounded answer uses only information from the context.
An ungrounded answer makes up information not in the context.

Retrieved Context: {retrieved_chunk}
LLM Answer: {llm_answer}

Return ONLY a JSON object in this exact format:
{{"score": 0.0, "reason": "one line explanation"}}

Score guide:
1.0 = answer is completely supported by context
0.7 = answer is mostly supported, minor additions
0.4 = answer uses some context but adds outside information
0.0 = answer is not supported by context at all"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    result = json.loads(response.choices[0].message.content)
    return result


def score_answer_relevance(question, llm_answer):
    """
    Score 3 — Answer Relevance
    Did the final answer actually address the question?
    Input: question + LLM answer
    Output: score 0 to 1
    """
    prompt = f"""You are an expert RAG pipeline evaluator.

Score how well the answer addresses the question asked.

Question: {question}
LLM Answer: {llm_answer}

Return ONLY a JSON object in this exact format:
{{"score": 0.0, "reason": "one line explanation"}}

Score guide:
1.0 = answer directly and completely addresses the question
0.7 = answer partially addresses the question
0.4 = answer is related but does not address the question
0.0 = answer is completely irrelevant to the question"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    result = json.loads(response.choices[0].message.content)
    return result


print("All 3 scoring functions ready!")
print("Score 1 — Context Relevance: did retrieval find the right chunk?")
print("Score 2 — Groundedness: did LLM answer from the context only?")
print("Score 3 — Answer Relevance: did the answer address the question?")

## Step 4 - Full RAG + Evaluation Pipeline
For each question we:
1. Retrieve the top chunk from FAISS
2. Generate an answer using the LLM
3. Score all 3 RAG Triad dimensions
4. Store everything in a results table

In [ ]:
def full_rag_with_evaluation(question):
    """
    Full RAG pipeline + RAG Triad evaluation in one function.
    Returns question, retrieved chunk, answer, and all 3 scores.
    """
    # STEP 1 — Retrieve top chunk from FAISS
    q_embedding = model.encode([question])
    q_float = np.array(q_embedding).astype("float32")
    faiss.normalize_L2(q_float)
    distances, indices = index.search(q_float, k=1)
    retrieved_chunk = chunks[indices[0][0]]
    retrieval_score = distances[0][0]

    # STEP 2 — Generate answer using LLM
    prompt = f"""You are a Swiggy delivery partner support agent.

Use the following policy context to answer the question.
Only use the context provided — do not make up information.
If the answer is not in the context, say "I could not find this information."

Policy Context: {retrieved_chunk}

Question: {question}

Answer clearly in 2 sentences."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    llm_answer = response.choices[0].message.content

    # STEP 3 — Score all 3 RAG Triad dimensions
    context_score = score_context_relevance(question, retrieved_chunk)
    ground_score = score_groundedness(retrieved_chunk, llm_answer)
    answer_score = score_answer_relevance(question, llm_answer)

    # STEP 4 — Return everything as one dictionary
    return {
        "question": question,
        "retrieved_chunk": retrieved_chunk,
        "llm_answer": llm_answer,
        "retrieval_score": round(retrieval_score, 4),
        "context_relevance": context_score["score"],
        "context_reason": context_score["reason"],
        "groundedness": ground_score["score"],
        "groundedness_reason": ground_score["reason"],
        "answer_relevance": answer_score["score"],
        "answer_reason": answer_score["reason"]
    }

print("Full RAG + Evaluation pipeline ready!")
print("Each question will now produce 3 quality scores automatically.")

## Step 5 - Run Full Evaluation
We evaluate 4 questions through the complete RAG Triad.
Each question gets 3 scores — Context Relevance, Groundedness, Answer Relevance.
This is how you prove your RAG pipeline works in a FAANG interview.

In [ ]:
# Our 4 test questions — one per policy section
test_questions = [
    "How much time do I have to raise a payment dispute?",
    "What should I do if a customer is not reachable?",
    "What happens if I use abusive language with a customer?",
    "How many days does it take to resolve a grievance?"
]

# Run full evaluation for each question
print("Running RAG Triad evaluation — this may take 30-60 seconds...")
print("Each question makes 4 API calls — RAG answer + 3 scores\n")

results = []
for i, question in enumerate(test_questions):
    print(f"Evaluating question {i+1} of {len(test_questions)}...")
    result = full_rag_with_evaluation(question)
    results.append(result)

print("\nEvaluation complete!")
print(f"Total questions evaluated: {len(results)}")

## Step 6 - Display RAG Triad Evaluation Results
We display all scores in a clean table using pandas.
This is exactly what you would show in a FAANG interview or a team review.

In [ ]:
import pandas as pd

# Build a clean results table
rows = []
for r in results:
    rows.append({
        "Question": r["question"][:50],
        "Retrieval Score": r["retrieval_score"],
        "Context Relevance": r["context_relevance"],
        "Groundedness": r["groundedness"],
        "Answer Relevance": r["answer_relevance"],
        "Avg RAG Score": round(
            (r["context_relevance"] + r["groundedness"] + r["answer_relevance"]) / 3, 3
        )
    })

# Create DataFrame
df = pd.DataFrame(rows)

# Print the table
print("=" * 90)
print("RAG TRIAD EVALUATION RESULTS — Swiggy Delivery Policy Bot")
print("=" * 90)
print(df.to_string(index=False))
print("=" * 90)

# Print overall pipeline averages
print(f"\nPIPELINE AVERAGES:")
print(f"Context Relevance : {df['Context Relevance'].mean():.3f}")
print(f"Groundedness      : {df['Groundedness'].mean():.3f}")
print(f"Answer Relevance  : {df['Answer Relevance'].mean():.3f}")
print(f"Overall RAG Score : {df['Avg RAG Score'].mean():.3f}")

## Step 7 - Detailed Scoring Reasons
We print the judge's reasoning for each score.
This helps us understand why scores are high or low
and identify where the pipeline needs improvement.

In [ ]:
# Print detailed breakdown for each question
for r in results:
    print(f"\nQuestion: '{r['question']}'")
    print(f"Retrieved chunk: '{r['retrieved_chunk'][:100]}...'")
    print(f"LLM Answer: '{r['llm_answer'][:100]}...'")
    print(f"")
    print(f"Context Relevance : {r['context_relevance']} — {r['context_reason']}")
    print(f"Groundedness      : {r['groundedness']} — {r['groundedness_reason']}")
    print(f"Answer Relevance  : {r['answer_relevance']} — {r['answer_reason']}")
    print("-" * 70)

## Step 8 - Improved Judge Prompt
We rewrite the scoring prompts to be more specific.
Better prompts = more consistent judge scores.
This shows iterative improvement — a key FAANG skill.

In [ ]:
def score_context_relevance_v2(question, retrieved_chunk):
    """
    Improved Context Relevance scorer.
    More specific instructions to reduce judge hallucination.
    """
    prompt = f"""You are evaluating a RAG pipeline.

Your job: does the retrieved context contain information that helps answer the question?

Question: {question}
Retrieved Context: {retrieved_chunk}

Rules:
- Score 1.0 if context contains a direct answer to the question
- Score 0.7 if context contains related information that partially helps
- Score 0.3 if context is loosely related
- Score 0.0 only if context is completely unrelated to the question topic

Do not penalize for what the context does NOT say.
Only score based on what IS present in the context.

Return ONLY this JSON — no other text:
{{"score": 0.0, "reason": "one line explanation"}}"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(response.choices[0].message.content)


def score_groundedness_v2(retrieved_chunk, llm_answer):
    """
    Improved Groundedness scorer.
    Minor additions like politeness words should not reduce score.
    """
    prompt = f"""You are evaluating a RAG pipeline.

Your job: is the LLM answer supported by the retrieved context?

Retrieved Context: {retrieved_chunk}
LLM Answer: {llm_answer}

Rules:
- Score 1.0 if all key facts in the answer come from the context
- Score 0.7 if most facts come from context with minor polite additions
- Score 0.3 if answer mixes context facts with outside information
- Score 0.0 only if answer completely ignores the context

Minor additions like "please", "our team", "for assistance" are acceptable.
Only penalize if the answer adds NEW FACTS not present in the context.

Return ONLY this JSON — no other text:
{{"score": 0.0, "reason": "one line explanation"}}"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(response.choices[0].message.content)


def score_answer_relevance_v2(question, llm_answer):
    """
    Improved Answer Relevance scorer.
    Judge must focus on whether the MAIN question was addressed.
    """
    prompt = f"""You are evaluating a RAG pipeline.

Your job: does the LLM answer address what the question is asking?

Question: {question}
LLM Answer: {llm_answer}

Rules:
- Score 1.0 if answer directly and completely addresses the question
- Score 0.7 if answer addresses the main point but misses minor details
- Score 0.3 if answer is related but does not directly address the question
- Score 0.0 only if answer is completely off topic

Focus on whether the MAIN question was answered.
Do not penalize for extra helpful information in the answer.

Return ONLY this JSON — no other text:
{{"score": 0.0, "reason": "one line explanation"}}"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(response.choices[0].message.content)


print("Improved scoring functions v2 ready!")


## Step 9 - Re-run Evaluation with Improved Judge
We run the same 4 questions with the v2 scoring functions.
Then we compare v1 vs v2 scores to measure improvement.
This is iterative evaluation — a core MLOps practice.

In [ ]:
def full_rag_with_evaluation_v2(question):
    """
    Full RAG pipeline + improved RAG Triad evaluation.
    Same RAG pipeline as before — only the judge prompts improved.
    """
    # STEP 1 — Retrieve top chunk from FAISS
    q_embedding = model.encode([question])
    q_float = np.array(q_embedding).astype("float32")
    faiss.normalize_L2(q_float)
    distances, indices = index.search(q_float, k=1)
    retrieved_chunk = chunks[indices[0][0]]
    retrieval_score = distances[0][0]

    # STEP 2 — Generate answer using LLM
    prompt = f"""You are a Swiggy delivery partner support agent.

Use the following policy context to answer the question.
Only use the context provided — do not make up information.
If the answer is not in the context, say "I could not find this information."

Policy Context: {retrieved_chunk}

Question: {question}

Answer clearly in 2 sentences."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    llm_answer = response.choices[0].message.content

    # STEP 3 — Score with improved v2 functions
    context_score = score_context_relevance_v2(question, retrieved_chunk)
    ground_score = score_groundedness_v2(retrieved_chunk, llm_answer)
    answer_score = score_answer_relevance_v2(question, llm_answer)

    return {
        "question": question,
        "retrieved_chunk": retrieved_chunk,
        "llm_answer": llm_answer,
        "retrieval_score": round(retrieval_score, 4),
        "context_relevance": context_score["score"],
        "context_reason": context_score["reason"],
        "groundedness": ground_score["score"],
        "groundedness_reason": ground_score["reason"],
        "answer_relevance": answer_score["score"],
        "answer_reason": answer_score["reason"]
    }

# Run evaluation with v2 judge
print("Running improved evaluation — this may take 30-60 seconds...")
results_v2 = []
for i, question in enumerate(test_questions):
    print(f"Evaluating question {i+1} of {len(test_questions)}...")
    result = full_rag_with_evaluation_v2(question)
    results_v2.append(result)

print("\nImproved evaluation complete!")

# Build comparison table — v1 vs v2
print("\n" + "=" * 80)
print("V1 vs V2 JUDGE COMPARISON")
print("=" * 80)
print(f"{'Question':<45} {'V1 Avg':>8} {'V2 Avg':>8} {'Change':>8}")
print("-" * 80)

for r1, r2 in zip(results, results_v2):
    v1_avg = round((r1["context_relevance"] + r1["groundedness"] + r1["answer_relevance"]) / 3, 3)
    v2_avg = round((r2["context_relevance"] + r2["groundedness"] + r2["answer_relevance"]) / 3, 3)
    change = round(v2_avg - v1_avg, 3)
    direction = "↑" if change > 0 else "↓" if change < 0 else "→"
    print(f"{r1['question'][:44]:<45} {v1_avg:>8} {v2_avg:>8} {direction} {change:>6}")

# Overall averages
v1_overall = sum((r["context_relevance"] + r["groundedness"] + r["answer_relevance"]) / 3 for r in results) / len(results)
v2_overall = sum((r["context_relevance"] + r["groundedness"] + r["answer_relevance"]) / 3 for r in results_v2) / len(results_v2)

print("-" * 80)
print(f"{'OVERALL PIPELINE SCORE':<45} {v1_overall:>8.3f} {v2_overall:>8.3f} {'↑' if v2_overall > v1_overall else '↓'} {round(v2_overall - v1_overall, 3):>6}")
print("=" * 80)

## What We Built
- Implemented the RAG Triad evaluation framework from scratch
- Built 3 LLM-as-Judge scoring functions: Context Relevance, Groundedness, Answer Relevance
- Used structured JSON output for reliable score parsing
- Ran evaluation on 4 Swiggy policy questions
- Diagnosed judge hallucination problems from the scoring reasons
- Improved judge prompts with clearer rules (v2)
- Compared v1 vs v2 scores using ablation study
- Overall pipeline score improved from 0.483 to 0.492

## Key Lessons Learned
- LLM-as-Judge is powerful but unreliable with small models
- Always print scoring REASONS not just scores — numbers without reasons are useless
- Use a stronger model as judge than the model generating answers
- Improvement is incremental — small gains per iteration add up over time
- Ablation study — change one thing at a time and measure impact

## RAG Triad Scores Explained
| Score              | What it measures                        | Target |
|--------------------|-----------------------------------------|--------|
| Context Relevance  | Did retrieval find the right chunk?     | > 0.7  |
| Groundedness       | Did LLM answer from context only?       | > 0.8  |
| Answer Relevance   | Did answer address the question?        | > 0.7  |
| Overall RAG Score  | Average of all three                    | > 0.7  |

## FAANG Interview Answer
"I evaluate RAG pipelines using the RAG Triad — Context Relevance measures
retrieval quality, Groundedness measures hallucination risk, and Answer Relevance
measures end-to-end quality. I use LLM-as-Judge with a stronger model than the
generator. I iterate on chunking, retrieval, and prompts until all three scores
are above 0.7. I track scores over time to detect degradation after knowledge
base updates."

## AWS Equivalent
| What we did here          | AWS service                              |
|---------------------------|------------------------------------------|
| LLM-as-Judge scoring      | Amazon Bedrock Model Evaluation          |
| Structured JSON output    | Amazon Bedrock with response format      |
| Evaluation results table  | Amazon CloudWatch metrics dashboard      |
| Iterative improvement     | Amazon SageMaker Experiments             |
| Production monitoring     | Amazon CloudWatch + SNS alerts           |

## What is Next
Module 3 is complete!
Module 4 — Fine-Tuning LLMs
We take a pre-trained model and train it further on Swiggy-specific data
so it understands our domain without needing RAG for every query.